# Lab 8: Entity Extraction Pipeline

In this lab, you will configure the entity extraction pipeline, add a manual entity to the knowledge graph, and explore how extraction settings control what gets stored. Entity extraction is what turns unstructured conversation text ("I love Christopher Nolan") into structured graph knowledge — named entities linked to conversations, preferences, and each other.

## What you will learn

- How to configure `MemorySettings` with extraction pipeline options
- How to manually add entities using `memory_client.long_term.add_entity()`
- How entity deduplication works with embedding similarity
- How to configure merge strategies and resolution thresholds

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

In [ ]:
from pydantic import SecretStr

from neo4j_agent_memory import MemoryClient, MemorySettings

## Configure Extraction Settings

The extraction pipeline uses three stages:

1. **spaCy** -- Fast, rule-based named entity recognition
2. **GLiNER** -- A lightweight model specialized for entity extraction
3. **LLM fallback** -- Uses a language model for entities the first two stages missed

Configure `MemorySettings` with `extractor_type: "pipeline"` to enable the multi-stage approach. Set a `confidence_threshold` to filter low-confidence results, and specify which `entity_types` to extract.

> **Note:** GLiNER is disabled in this workshop (`enable_gliner: False`) because it downloads a ~500MB model from HuggingFace on first use, which is impractical with many participants on shared network bandwidth. The pipeline still works well with spaCy + LLM fallback. In production, you can enable GLiNER for improved extraction accuracy.

In [ ]:
settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    extraction={
        "extractor_type": "pipeline",
        "enable_spacy": True,
        "enable_gliner": False,  # Disabled: downloads ~500MB model from HuggingFace, impractical in a workshop
        "enable_llm_fallback": True,
        "confidence_threshold": 0.5,
        "llm_temperature": 1.0,  # gpt-5-mini only supports temperature=1.0
        "entity_types": [
            "PERSON", "ORGANIZATION", "LOCATION", "EVENT", "OBJECT"
        ],
    },
)
print("Settings configured")

## Connect the Memory Client

In [ ]:
memory_client = MemoryClient(settings)
await memory_client.connect()
print("Connected to Neo4j")

## Add a Manual Entity

While entity extraction typically runs automatically during conversations (via `extract_entities=True` on `Neo4jMicrosoftMemory`), you can also add entities manually. Entities become labeled nodes in the Neo4j knowledge graph, linked to their source conversations — this is how unstructured conversation text becomes structured graph data.

Use `memory_client.long_term.add_entity()` to add an entity with a name, type, and description. The entity gets a vector embedding and is checked for duplicates automatically.

In [ ]:
entity, dedup_result = await memory_client.long_term.add_entity(
    name="Inception",
    entity_type="OBJECT",
    description="2010 science fiction film directed by Christopher Nolan",
)
print(f"Added entity: name={entity.name}, type={entity.entity_type}, id={entity.id}")

## Test Deduplication

Add the same entity again. The deduplication system uses embedding similarity to detect that "Inception" already exists and merges the entries rather than creating a duplicate.

In [ ]:
duplicate, dedup_result = await memory_client.long_term.add_entity(
    name="Inception",
    entity_type="OBJECT",
    description="A film about dream infiltration by Christopher Nolan",
)
print(f"Result: name={duplicate.name}, id={duplicate.id}")
print(f"Same entity? {duplicate.id == entity.id}")

## Merge Strategies

When multiple pipeline stages find the same entity, the `merge_strategy` controls which result to keep. Choose **confidence** for highest quality, **union** for maximum coverage, **intersection** when you only want high-certainty entities found by multiple stages, or **cascade** for speed with fallback to later stages.

- **`confidence`** -- Keeps the highest confidence score per entity (default)
- **`union`** -- Keeps all unique entities from all stages
- **`intersection`** -- Only keeps entities found by multiple stages
- **`cascade`** -- Uses first stage results, fills gaps with later stages

In [ ]:
settings_with_merge = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    extraction={
        "extractor_type": "pipeline",
        "merge_strategy": "confidence",
    },
)
print(f"Merge strategy: {settings_with_merge.extraction.merge_strategy}")

## Resolution Settings

Entity deduplication uses embedding similarity with configurable thresholds:

- **`exact_threshold`** (1.0) -- Identical names are always merged
- **`fuzzy_threshold`** (0.85) -- Similar names above this threshold are merged
- **`semantic_threshold`** (0.8) -- Semantically similar entities above this threshold are merged

In [ ]:
settings_with_resolution = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    resolution={
        "strategy": "composite",
        "exact_threshold": 1.0,
        "fuzzy_threshold": 0.85,
        "semantic_threshold": 0.8,
    },
)
print(f"Resolution strategy: {settings_with_resolution.resolution.strategy}")

## Close the Connection

In [ ]:
await memory_client.close()
print("Connection closed")

## Summary

The entity extraction pipeline uses three stages (spaCy, GLiNER, LLM) to extract structured entities from conversation text. Results are merged using configurable strategies and deduplicated using embedding similarity. Extracted entities become labeled nodes in Neo4j linked to their source conversations.

In this workshop, GLiNER was disabled to avoid downloading a large model over shared bandwidth. In production, enabling all three stages provides the best extraction coverage.

**What's next:** In the next lab, you will add memory tools that give the agent explicit control over its memory, combining passive injection with active search, save, and recall operations.